# Lab 4: Tokenization and Text Processing - The Language Interface

## Lab Overview

This lab explores tokenization, the crucial process that converts human text into numerical representations that neural networks can understand. You'll learn about different tokenization strategies, encoding/decoding processes, and their impact on model performance.

## Learning Objectives

By the end of this lab, you will:
- Understand different tokenization approaches (word, subword, character)
- Master encoding and decoding processes for text
- Learn about vocabulary management and token IDs
- Explore multilingual tokenization challenges
- Set up AMD GPU backend for efficient text processing
- Connect tokenization to transformer model inputs

---

In [ ]:
# AMD GPU initialization


# Essential Imports and GPU Setup
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, pipeline, AutoModel
import numpy as np

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 1. Tokenizer Setup - Using GPT-2 for Universal Access

print("=== Setting up Tokenizer ===")

# Use GPT-2 as it's widely available and demonstrates key concepts
model_name = 'gpt2'

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"Successfully loaded tokenizer: {model_name}")
    print(f"Vocabulary size: {len(tokenizer)}")
    print(f"Model max length: {tokenizer.model_max_length}")

    # Add padding token if it doesn't exist (GPT-2 doesn't have one by default)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print("Added padding token (using EOS token)")

except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("Please ensure you have internet connection or the model is cached locally")

## 1. Basic Tokenization Concepts

Understanding how text is converted to numbers that neural networks can process.

In [ ]:
# 1.1 Text to Token Conversion

print("=== Basic Tokenization Examples ===")

# Test sentences with different characteristics
test_texts = [
    "Hello, world!",                    # Simple English
    "The quick brown fox jumps.",       # Common English words
    "Machine learning is fascinating.", # Technical terms
    "OpenAI's GPT-2 model",            # Mixed case, apostrophes
    "2023年人工智能发展",                 # Mixed languages/characters
    "COVID-19 pandemic affected everyone.", # Numbers, acronyms
]

for i, text in enumerate(test_texts, 1):
    print(f"\nExample {i}: '{text}'")

    # Tokenize the text
    tokens = tokenizer.tokenize(text)
    print(f"  Tokens: {tokens}")
    print(f"  Number of tokens: {len(tokens)}")

    # Convert to IDs
    token_ids = tokenizer.encode(text)
    print(f"  Token IDs: {token_ids}")

    # Decode back to text
    decoded = tokenizer.decode(token_ids)
    print(f"  Decoded: '{decoded}'")
    print(f"  Perfect reconstruction: {text == decoded.strip()}")

print(f"\nKey Observations:")
print(f"- Punctuation often becomes separate tokens")
print(f"- Subword tokenization breaks unknown/rare words")
print(f"- Numbers and special characters need special handling")
print(f"- Different languages may tokenize differently")

In [ ]:
# 1.2 Multilingual Tokenization Analysis

print("=== Multilingual Text Processing ===")

# Test with different languages
multilingual_texts = {
    "English": "Shanghai is a famous city in China.",
    "Chinese": "上海是中国一座有名的城市。",
    "Mixed": "上海科技大学 (ShanghaiTech University) is located in China.",
    "Technical": "NLP (Natural Language Processing) 自然语言处理"
}

for lang, text in multilingual_texts.items():
    print(f"\n{lang} Text: '{text}'")

    # Detailed tokenization analysis
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text, return_tensors="pt")

    print(f"  Tokens ({len(tokens)}): {tokens}")
    print(f"  Token IDs: {token_ids[0].tolist()}")

    # Analyze each token individually
    for i, (token, token_id) in enumerate(zip(tokens, tokenizer.encode(text))):
        decoded_single = tokenizer.decode([token_id])
        print(f"    [{i}] '{token}' -> ID {token_id} -> '{decoded_single}'")

    # Check decoding accuracy
    decoded_full = tokenizer.decode(token_ids[0])
    print(f"  Reconstructed: '{decoded_full}'")

    # Calculate efficiency
    char_count = len(text)
    token_count = len(tokens)
    compression_ratio = char_count / token_count if token_count > 0 else 0
    print(f"  Compression: {char_count} chars -> {token_count} tokens (ratio: {compression_ratio:.2f})")

print(f"\nTokenization Insights:")
print(f"- English words often map to single tokens")
print(f"- Chinese characters may need multiple tokens")
print(f"- Mixed language text requires careful handling")
print(f"- Compression ratio varies by language and content")

### Understanding the � Replacement Character

### What You're Seeing

When decoding individual tokens, you might see:

```
'åį' -> ID 39355 -> '�'
```

The � character appears because **GPT-2 uses Byte-Level BPE**, which splits characters into bytes.


### Why This Happens

**Multi-byte characters need multiple tokens:**

- English "A" = 1 byte = usually 1 token ✓
- Chinese "大" = 3 bytes = typically 2-3 tokens

**Single token = incomplete character:**

- Decoding one token alone → � (incomplete byte sequence)
- Decoding all tokens together → "大" (complete character) ✓

### Key Takeaway

 **Always decode complete token sequences**, not individual tokens

 **This is expected behavior**, not an error

 **Chinese text uses ~3x more tokens** than English for the same meaning



## 1.3 Tensor Processing and Batch Operations

In real-world NLP tasks, we rarely process sentences one by one. To leverage the parallel processing power of GPUs, we group multiple sequences into a **batch**. However, because sentences have different lengths, we must transform them into a uniform shape—a rectangular **Tensor**.

### Core Concepts

* **Padding:** Since tensors must have fixed dimensions, shorter sentences are padded with a special token (usually `[PAD]`) to match the length of the longest sentence in the batch.
* **Truncation:** If a sentence exceeds the model's maximum length (e.g., 512 tokens for BERT), it is cut down to fit.
* **Attention Masks:** These are binary tensors that tell the model which tokens are "real" (1) and which are just padding (0), ensuring the model ignores the empty space.
* **Device Placement:** For high-performance computing, we move these tensors from the CPU to a hardware accelerator like a **GPU (CUDA)**.



### Implementation Guide

To complete the `TODO` sections in your code, keep these functions in mind:

1.  **Batch Tokenization:** When calling the `tokenizer`, set `padding=True`, `truncation=True`, and `return_tensors="pt"` (for PyTorch).
2.  **Device Transfer:** Use the `.to(device)` method to move your tensors to the GPU. For example: `batch_input_ids.to("cuda")`.
3.  **Tensor Math:** To find the number of real tokens, you can use `.sum()` on the `attention_mask`. Since "1" represents a real token, summing the mask gives you the count of non-padded elements.



In [ ]:
# 1.3 Tensor Processing and Batch Operations
TODO = None
print("=== Working with Token Tensors ===")

# Sample texts for batch processing
sample_texts = [
    "Artificial intelligence is transforming the world.",
    "Machine learning models need large datasets.",
    "Deep learning uses neural networks with many layers.",
]

# --- Step 1: Batch Tokenization ---
# TODO: Configure the tokenizer call to pad, truncate, and return PyTorch tensors.
print("Running batch processing...")
batch_encoded = tokenizer(
    sample_texts, padding=TODO, truncation=TODO, return_tensors="pt"
)

batch_input_ids = batch_encoded["input_ids"]
batch_attention_mask = batch_encoded["attention_mask"]

print(f"Batch input IDs shape: {batch_input_ids.shape}")
print(f"Batch attention mask shape: {batch_attention_mask.shape}")


# --- Step 2: Move Tensors to Device ---
# TODO: Move the input_ids and attention_mask tensors to the designated device.
if torch.cuda.is_available():
    batch_input_ids = TODO
    batch_attention_mask = TODO
    print(f"Batch tensors successfully moved to device: {batch_input_ids.device}")
else:
    print("CUDA not available. Tensors remain on CPU.")


# --- Step 3: Analyze Padding ---
# TODO: Loop through the batch and calculate the number of real vs. padding tokens for each sample.
print("\nAnalyzing padding per sample:")
for i in range(len(sample_texts)):
    # Hint: The attention mask contains 1s for real tokens and 0s for padding.
    # You can use tensor operations like .sum() to count them.

    # Get the mask for the current sample
    sample_mask = batch_attention_mask[i]

    # Calculate counts
    actual_tokens_count = sample_mask.sum().item()
    total_tokens_count = len(sample_mask)
    padded_tokens_count = TODO

    print(
        f"  Sample {i+1}: {actual_tokens_count} real tokens, {padded_tokens_count} padding tokens"
    )
print(f"\nKey Concepts:")
print(f"- Padding enables batch processing of variable-length sequences")
print(f"- Attention masks tell the model which tokens are real vs padding")
print(f"- GPU processing accelerates tokenization for large batches")
print(f"- Truncation handles sequences longer than model limits")

In [ ]:
# 2. Tokenization Methods Comparison

print("=== Different Tokenization Approaches ===")

test_text = "The preprocessing step tokenizes input text efficiently."

print(f"Input text: '{test_text}'")
print(f"Character count: {len(test_text)}")

# Method 1: Direct tokenization
print(f"\n1. Direct Tokenization:")
tokens = tokenizer.tokenize(test_text)
print(f"  Tokens: {tokens}")
print(f"  Token count: {len(tokens)}")

# Method 2: Encoding to IDs
print(f"\n2. Encoding to IDs:")
ids = tokenizer.encode(test_text)
print(f"  Token IDs: {ids}")
print(f"  ID count: {len(ids)}")

# Method 3: Full encoding with special tokens
print(f"\n3. Full Encoding (with special tokens):")
full_encoded = tokenizer.encode(test_text, add_special_tokens=True)
print(f"  With special tokens: {full_encoded}")
print(f"  Special token IDs: start={tokenizer.bos_token_id}, end={tokenizer.eos_token_id}")

# Method 4: Token to ID conversion
print(f"\n4. Token to ID Conversion:")
token_to_id_result = tokenizer.convert_tokens_to_ids(tokens)
print(f"  Converted IDs: {token_to_id_result}")
print(f"  Matches direct encoding: {token_to_id_result == tokenizer.encode(test_text, add_special_tokens=False)}")

# Method 5: ID to Token conversion
print(f"\n5. ID to Token Conversion:")
id_to_token_result = tokenizer.convert_ids_to_tokens(ids)
print(f"  Converted tokens: {id_to_token_result}")

# Analyze special tokens
print(f"\n6. Special Token Analysis:")
special_tokens = {
    'PAD': tokenizer.pad_token_id,
    'UNK': tokenizer.unk_token_id,
    'BOS': tokenizer.bos_token_id,
    'EOS': tokenizer.eos_token_id,
}

for name, token_id in special_tokens.items():
    if token_id is not None:
        token_str = tokenizer.decode([token_id])
        print(f"  {name} token: ID {token_id} -> '{token_str}'")
    else:
        print(f"  {name} token: Not defined")

print(f"\nTokenization Workflow:")
print(f"  Text -> tokenize() -> Tokens -> convert_tokens_to_ids() -> IDs")
print(f"  Text -> encode() -> IDs (direct)")
print(f"  IDs -> decode() -> Text (reconstruction)")
print(f"  IDs -> convert_ids_to_tokens() -> Tokens -> detokenize() -> Text")



## 2.1 Vocabulary Analysis and Token Statistics

Every Large Language Model (LLM) has a fixed "dictionary" called a **Vocabulary**. Before a model can process text, it must break words down into **Token IDs** that correspond to entries in this vocabulary. Analyzing the vocabulary helps us understand how a model perceives everything from punctuation to complex numbers.

### Core Concepts

* **Vocabulary Size:** This is the total number of unique tokens the model knows. A larger vocabulary (e.g., 50,257 for GPT-2) allows for more efficient encoding but requires more memory.
* **Token ID Mapping:** Each token (a word, a subword like "ing", or a character) is mapped to a unique integer. 
* **Subword Tokenization:** Modern models don't just store whole words. They use algorithms like **Byte Pair Encoding (BPE)** to break rare words into smaller, meaningful chunks.
    * *Example:* "tokenization" might be split into `["token", "ization"]`.
* **Special Characters & Numbers:** Models often treat symbols and digits differently. Some represent "123" as a single token, while others split it into `["1", "2", "3"]`.



### Implementation Guide

To fill in the `TODO` placeholders, you will use these common `tokenizer` methods:

1.  **Finding Size:** Use `len(tokenizer)` or `tokenizer.vocab_size` to get the total count.
2.  **Decoding IDs:** Use `tokenizer.decode([token_id])` to turn a number back into a human-readable string.
3.  **Encoding Text:** Use `tokenizer.tokenize(text)` to see the string chunks, and `tokenizer.convert_tokens_to_ids()` or `tokenizer.encode(text, add_special_tokens=False)` to get the numerical IDs.
4.  **Length Analysis:** When calculating `token_text`, remember that many tokens include a special character (like `Ġ` in GPT-2) to represent a leading space.



In [ ]:
# 2.1 Vocabulary Analysis and Token Statistics

print("=== Tokenizer Vocabulary Analysis ===")

# Vocabulary statistics
# Hint: The tokenizer object has a method or attribute that provides the total vocabulary size.
# For example, for GPT-2, you can access it via len(tokenizer) or tokenizer.vocab_size.
vocab_size = TODO
print(f"Total vocabulary size: {vocab_size:,} tokens")

# Sample vocabulary entries
print(f"\nSample vocabulary entries:")
sample_ids = [0, 1, 2, 100, 1000, vocab_size-1]
for token_id in sample_ids:
    if token_id < vocab_size:
        token = tokenizer.decode([TODO])
        print(f"  ID {token_id:>6}: '{token}'")

# Most common tokens (usually shorter, more frequent)
print(f"\nFirst 20 tokens (typically most common):")
# Hint: You can decode token IDs to get their string representation. For example, tokenizer.decode([token_id]) will give you the token string for that ID.
first_tokens = [(i, TODO) for i in range(min(20, vocab_size))]
for token_id, token in first_tokens:
    print(f"  {token_id:>2}: '{token}'")

# Token length analysis
print(f"\nToken length analysis:")
sample_size = min(1000, vocab_size)
token_lengths = []
for i in range(sample_size):
    # Hint: You can decode token IDs to get their string representation and calculate its length.
    token_text = TODO
    token_lengths.append(len(token_text))

if token_lengths:
    avg_length = sum(token_lengths) / len(token_lengths)
    max_length = max(token_lengths)
    min_length = min(token_lengths)

    print(f"  Sample size: {sample_size} tokens")
    print(f"  Average token length: {avg_length:.2f} characters")
    print(f"  Token length range: {min_length} - {max_length} characters")

# Special character handling
print(f"\nSpecial character analysis:")
special_chars = ['!', '@', '#', '$', '%', '&', '*', '(', ')', '-', '_', '=', '+']
for char in special_chars:
    # Hint: You can tokenize special characters to see how they are represented in the vocabulary.
    # For example, tokenizer.tokenize(char) will show you how the character is tokenized, and tokenizer.convert_tokens_to_ids() can give you the corresponding IDs.
    tokens = TODO
    # You can use tokenizer.encode(char, add_special_tokens=False) to get the ID directly.
    ids = TODO
    print(f"  '{char}' -> tokens: {tokens} -> IDs: {ids}")

# Number handling
print(f"\nNumber tokenization:")
numbers = ['0', '123', '2023', '3.14', '1,000,000']
for num in numbers:
    # Hint: Similar to special characters, you can tokenize numbers to see how they are represented in the vocabulary.
    tokens = TODO
    print(f"  '{num}' -> {len(tokens)} tokens: {tokens}")

print(f"\nVocabulary Insights:")
print(f"- Smaller IDs often represent more frequent tokens")
print(f"- Special characters may become individual tokens")
print(f"- Numbers can be split into multiple tokens")
print(f"- Subword tokenization handles rare/unknown words")
print(f"- Vocabulary size affects model memory and computation")

## 3. Advanced Tokenization Topics

Exploring advanced concepts like subword algorithms, out-of-vocabulary handling, and tokenization strategies for different domains.

In [ ]:
# 3.1 Subword Tokenization Analysis

print("=== Subword Tokenization Behavior ===")

# Test with various word types
test_words = [
    # Common words (likely single tokens)
    "the", "and", "is", "to", "in",

    # Uncommon/rare words (likely split)
    "preprocessing", "tokenization", "subword", "transformer",

    # Technical terms
    "PyTorch", "GPU", "CUDA", "neural", "networks",

    # Made-up words
    "superdupertokenizer", "pseudotechnical", "multidimensional",

    # Words with affixes
    "running", "walked", "happiness", "beautiful", "quickly"
]

def analyze_tokenization(word_list, tokenizer):
    """
    Analyzes and categorizes a list of words based on their tokenization by GPT-2.

    Args:
        word_list (list): A list of strings to analyze.
        tokenizer: An initialized Hugging Face tokenizer.

    Returns:
        tuple: A tuple containing:
            - categorized_words (dict): A dictionary grouping words by their token count.
            - most_fragmented_word (dict): A dictionary with the most fragmented word and its token count.
    """
    categorized_words = {}
    most_fragmented_word = {"word": "", "count": 0}

    # --- Step 1 & 2: Iterate and Categorize ---
    for word in word_list:
        # TODO: Tokenize the current word to get its subword tokens.
        # Note: Do not add special tokens like [CLS] or [SEP].
        tokens = TODO

        # TODO: Get the number of tokens.
        num_tokens = TODO

        # TODO: Populate the categorized_words dictionary.
        # If the key (num_tokens) doesn't exist, create it with a new list.
        # Then, append the current word to the correct list.
        if num_tokens not in categorized_words:
            categorized_words[num_tokens] = []
        categorized_words[num_tokens].append(TODO)

        # --- Step 3: Find the Most Fragmented Word ---
        # TODO: Check if the current word has more tokens than the one stored
        # in most_fragmented_word. If so, update the dictionary.
        if num_tokens > most_fragmented_word["count"]:
            most_fragmented_word["word"] = TODO
            most_fragmented_word["count"] = TODO

    # --- Step 4: Return the completed data structures ---
    return categorized_words, most_fragmented_word

# --- Execution ---
# Call the function and store the results.
analysis_results, most_split_word = analyze_tokenization(test_words, tokenizer)

# --- Verification ---
# The following print statements will help you verify your results.
print("--- Word Categorization by Token Count ---")
for count, words in sorted(analysis_results.items()):
    print(f"{count} Tokens: {words}")

print("\n--- Most Fragmented Word ---")
print(f"Word: '{most_split_word['word']}' -> {most_split_word['count']} tokens")

print(f"\nSubword Algorithm Benefits:")
print(f"- Handles infinite vocabulary with finite token set")
print(f"- Breaks unknown words into recognizable parts")
print(f"- Balances vocabulary size vs token sequence length")
print(f"- Enables cross-lingual transfer (shared subwords)")
print(f"- Reduces out-of-vocabulary problems")



## 3.2 Tokenization Performance and Efficiency

When building production-level AI applications, raw accuracy isn't the only metric that matters—**latency** and **throughput** are just as critical. This section explores how the tokenizer scales as text grows longer and why "batching" is the secret sauce for high-performance NLP.

### Core Concepts

* **Latency vs. Throughput:** * **Latency** is how long it takes to process a single string (measured in milliseconds). 
    * **Throughput** is how many total tokens/sentences you can process per second.
* **The Power of Batching:** Processing 100 sentences one-by-one in a `for` loop is significantly slower than passing a list of 100 sentences to the `tokenizer()` at once. This is because batching allows the library to use optimized C++ or Rust backends and parallelize the work across your CPU or GPU cores.
* **Characters-to-Token Ratio:** This ratio (usually around 3.5 to 4 characters per token for English) helps you estimate how much memory a text file will occupy once it is converted into a tensor.
* **Memory Footprint:** A Python list of strings is flexible but memory-heavy. Once we convert tokens into **PyTorch Tensors**, they are stored as contiguous blocks of integers, which is much more memory-efficient and ready for mathematical operations.



### What to Watch For in the Results

1.  **Linear Scaling:** Notice how the `Encode` time usually grows linearly with the number of characters.
2.  **The Batching "Speedup":** Pay close attention to the `Speedup` calculation. Even on a standard CPU, batching is almost always faster because it reduces the overhead of repeatedly calling Python functions.
3.  **GPU vs. CPU:** If you have a GPU enabled, moving the final `tensor_ids` to the device (`.to("cuda")`) is the final step before the model can perform a "forward pass" to generate text or classifications.



In [ ]:
# 3.2 Tokenization Performance and Efficiency

print("=== Tokenization Performance Analysis ===")

import time

# Test with different text lengths
test_texts = [
    "Short text.",
    "This is a medium length sentence with several words to tokenize and process efficiently.",
    " ".join(["This is a much longer text that repeats the same content multiple times."] * 10),
    " ".join(["Very long text with repeated content for performance testing."] * 50)
]

print("Performance comparison:")
for i, text in enumerate(test_texts):
    char_count = len(text)

    # Time the tokenization
    start_time = time.time()
    tokens = tokenizer.tokenize(text)
    tokenize_time = time.time() - start_time

    # Time the encoding
    start_time = time.time()
    ids = tokenizer.encode(text)
    encode_time = time.time() - start_time

    # Time the decoding
    start_time = time.time()
    decoded = tokenizer.decode(ids)
    decode_time = time.time() - start_time

    print(f"\nText {i+1} ({char_count} chars, {len(tokens)} tokens):")
    print(f"  Tokenize: {tokenize_time*1000:.2f}ms")
    print(f"  Encode:   {encode_time*1000:.2f}ms")
    print(f"  Decode:   {decode_time*1000:.2f}ms")
    print(f"  Chars/token ratio: {char_count/len(tokens):.2f}")

# Batch vs individual processing
print(f"\nBatch vs Individual Processing:")
batch_texts = ["Sample text number " + str(i) for i in range(100)]

# Individual processing
start_time = time.time()
individual_results = [tokenizer.encode(text) for text in batch_texts]
individual_time = time.time() - start_time

# Batch processing
start_time = time.time()
batch_result = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
batch_time = time.time() - start_time

print(f"  Individual processing: {individual_time*1000:.2f}ms for {len(batch_texts)} texts")
print(f"  Batch processing: {batch_time*1000:.2f}ms for {len(batch_texts)} texts")
print(f"  Speedup: {individual_time/batch_time:.2f}x faster with batching")

# Memory usage analysis
print(f"\nMemory Usage Analysis:")
sample_text = "This is a sample text for memory analysis. " * 20

# Create different representations
tokens = tokenizer.tokenize(sample_text)
ids = tokenizer.encode(sample_text)
tensor_ids = tokenizer(sample_text, return_tensors="pt")["input_ids"]

print(f"  Original text: {len(sample_text)} characters")
print(f"  Token list: {len(tokens)} tokens (Python list)")
print(f"  ID list: {len(ids)} IDs (Python list)")
print(f"  Tensor: {tensor_ids.shape} (PyTorch tensor)")

if torch.cuda.is_available():
    gpu_tensor = tensor_ids.to(device)
    print(f"  GPU tensor: {gpu_tensor.shape} on {gpu_tensor.device}")

print(f"\nEfficiency Insights:")
print(f"- Batch processing significantly improves throughput")
print(f"- GPU tensors enable parallel processing")
print(f"- Subword tokenization balances vocabulary size and sequence length")
print(f"- Caching tokenizer results can improve repeated processing")
print(f"- Memory usage scales with sequence length and batch size")

## 4. Tokenization for LLM Applications

Understanding how tokenization impacts Large Language Model training, inference, and performance.



**Your Task**:

1.  **Implement the `create_llm_prompt` function**: This function will programmatically construct a formatted prompt from three distinct parts: a system message, a user query, and a long context string.
2.  **Format the Prompt**: The final prompt must follow this exact structure, including the special tokens: `<|system|>{system_message}<|endoftext|><|user|>{context}\n\n{user_query}<|endoftext|><|assistant|>`
3.  **Tokenize and Analyze**: Tokenize the final combined prompt as well as each individual component (`system_message`, `context`, `user_query`).
4.  **Perform Context Window Analysis**: Calculate the total number of tokens in the final prompt. Determine if this total exceeds the `max_context_length`.
5.  **Calculate Component Contribution**: For each component, calculate what percentage of the *total* prompt's token count it occupies.
6.  **Return Structured Data**: The function must return a dictionary containing all the calculated metrics: the final prompt string, the token counts for each part and the total, a boolean indicating if the prompt is within the context limit, and the percentage contribution of each part.


In [ ]:
# 4.1 LLM Tokenization Patterns and Best Practices

print("=== LLM Tokenization Patterns ===")


# Add special tokens that might be used in instruction-tuned models.
# This ensures they are treated as single tokens.
special_tokens_dict = {'additional_special_tokens': ['<|system|>', '<|user|>', '<|assistant|>']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

# --- Data ---
# Define the components for our prompt.
system_message = "You are a helpful assistant that summarizes technical documents."
user_query = "Based on the provided text, what is the main innovation described?"
long_context = "Tokenization is the process of breaking down a stream of text into smaller units called tokens. These tokens can be words, subwords, or characters. For Large Language Models (LLMs), subword tokenization, like Byte-Pair Encoding (BPE), is common. It balances vocabulary size and the ability to handle unknown words. " * 8
max_window_size = 256 # A simulated small context window for testing.


def create_llm_prompt(system_message, user_query, context, tokenizer, max_context_length):
    """
    Builds and analyzes a formatted LLM prompt.

    Args:
        system_message (str): The system-level instruction for the model.
        user_query (str): The user's direct question.
        context (str): The context or document the query is about.
        tokenizer: An initialized Hugging Face tokenizer.
        max_context_length (int): The maximum number of tokens allowed.

    Returns:
        dict: A dictionary containing the full prompt and its analysis.
    """

    # --- Step 1 & 2: Format the Prompt ---
    # Construct the final prompt string using an f-string and the specified format.
    final_prompt = f"<|system|>{system_message}<|endoftext|><|user|>{context}\n\n{user_query}<|endoftext|><|assistant|>"

    # --- Step 3: Tokenize and Analyze ---
    # TODO: Tokenize the final prompt and each individual component.
    # Use add_special_tokens=False for components to get their raw token count.
    prompt_tokens = tokenizer.encode(final_prompt)
    system_tokens = tokenizer.encode(TODO, add_special_tokens=False)
    query_tokens = tokenizer.encode(TODO, add_special_tokens=False)
    context_tokens = tokenizer.encode(TODO, add_special_tokens=False)

    total_token_count = len(prompt_tokens)
    system_token_count = len(system_tokens)
    query_token_count = len(query_tokens)
    context_token_count = len(context_tokens)

    # --- Step 4: Context Window Analysis ---
    # TODO: Check if the total token count exceeds the max_context_length.
    is_within_limit = TODO

    # --- Step 5: Calculate Component Contribution ---
    # TODO: Calculate the percentage of the total tokens used by each part.
    # Handle the case where total_token_count might be zero.
    system_percentage = (system_token_count / total_token_count * 100) if total_token_count > 0 else 0
    query_percentage = TODO
    context_percentage = TODO

    # --- Step 6: Return Structured Data ---
    analysis = {
        "final_prompt": final_prompt,
        "total_tokens": total_token_count,
        "is_within_limit": is_within_limit,
        "component_analysis": {
            "system": {"tokens": system_token_count, "percentage": system_percentage},
            "user_query": {"tokens": query_token_count, "percentage": query_percentage},
            "context": {"tokens": context_token_count, "percentage": context_percentage},
        }
    }

    return analysis


# --- Execution ---
prompt_analysis = create_llm_prompt(system_message, user_query, long_context, tokenizer, max_window_size)

# --- Verification ---
print(f"--- Prompt Analysis (Max Window: {max_window_size}) ---")
print(f"Total Tokens: {prompt_analysis['total_tokens']}")
print(f"Fits in Context Window: {prompt_analysis['is_within_limit']}")
print("\n--- Component Breakdown ---")
for component, data in prompt_analysis['component_analysis'].items():
    print(f"  - {component.capitalize()}: {data['tokens']} tokens ({data['percentage']:.2f}%)")

print(f"\nLLM Tokenization Best Practices:")
print(f"- Monitor token counts to stay within model limits")
print(f"- Use appropriate special tokens for your model")
print(f"- Consider token efficiency when designing prompts")
print(f"- Batch process multiple inputs for efficiency")
print(f"- Handle different languages and domains appropriately")
print(f"- Account for tokenization differences between models")
print(f"- Use attention masks properly for padded sequences")
print(f"- Consider subword boundary effects on model understanding")

## Summary and Key Takeaways

**What You've Learned:**

1. **Tokenization Fundamentals**: How text is converted to numerical representations
2. **Multiple Approaches**: Direct tokenization, encoding, decoding, and batch processing
3. **Multilingual Handling**: Challenges and solutions for different languages
4. **Vocabulary Management**: Understanding token IDs, special tokens, and vocabulary size
5. **Subword Algorithms**: How models handle unknown words and achieve vocabulary efficiency
6. **Performance Optimization**: Batch processing, GPU acceleration, and memory management
7. **LLM Applications**: Real-world patterns and best practices for language models

**Critical Concepts for LLM Development:**
- Tokenization is the bridge between human language and neural networks
- Subword tokenization enables handling of infinite vocabulary with finite tokens
- Proper padding and attention masks are essential for batch processing
- Token efficiency affects model performance and computational costs
- Different models may require different tokenization strategies

**Next Steps:**
- Experiment with different tokenizers (BERT, T5, LLaMA)
- Explore domain-specific tokenization challenges
- Learn about custom vocabulary creation and adaptation
- Understand tokenization impact on model fine-tuning
- Study multilingual and cross-lingual tokenization strategies